### Installing Required Libraries

In [55]:
!pip install transformers langchain-core langchain-community langchain-ollama langchain-openai langchain-huggingface pydantic faiss-cpu rank_bm25
!apt-get update -qq
!apt-get install -y -qq zstd

!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [62]:
import os
import getpass
from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)

# Set your OpenAI API Key here if you have one. This is optional.
openai_key = os.getenv("OPENAI_API_KEY") or getpass.getpass("Enter your OpenAI API Key (leave empty to skip): ")
if openai_key:
    os.environ["OPENAI_API_KEY"] = openai_key

print("API key input prompts added.")

Enter your OpenAI API Key (leave empty to skip): ··········
API key input prompts added.


<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin-right: 20px; height: 55px" height="55px">

# 1. Architecting Robust RAG Pipelines (Solved Reference)

Fully worked solutions to all three exercises. Use this to check your work or to
catch up if you fell behind during the live lab. See `solutions folder` for
prose explanations alongside this code.

---

## Solution Guide

1. [Exercise 1 — Query Router with Range Detection](#exercise1)
2. [Exercise 2 — Jurisdiction-Aware Legal Pipeline](#exercise2)
3. [Exercise 3 — Tuning the Pipeline Back Within Budget](#exercise3)

---

<h2 id="exercise1"> Exercise 1 — Query Router with Range Detection </h2>

In [14]:
import re

ALPHANUMERIC_ID_PATTERN = re.compile(r"[A-Z]{1,5}[-#]\d{2,}[A-Za-z0-9-]*")
QUOTED_PHRASE_PATTERN = re.compile(r'"[^"]+"')

RELATIVE_DAYS_PATTERN = re.compile(
    r"(?:older than|more than|in the last|within the last|past)\s+(\d+)\s+(day|days|week|weeks|month|months)",
    re.IGNORECASE,
)
BETWEEN_DATES_PATTERN = re.compile(
    r"between\s+([A-Za-z]+)\s+and\s+([A-Za-z]+)", re.IGNORECASE
)

def route_query_v2(query: str) -> dict:
    relative_match = RELATIVE_DAYS_PATTERN.search(query)
    if relative_match:
        value, unit = relative_match.groups()
        operator = "gt" if any(w in query.lower() for w in ["older than", "more than", "past"]) else "lte"
        return {
            "sparse_weight": 0.30,
            "dense_weight": 0.70,
            "reason": "relative_range",
            "requires_prefilter": True,
            "filter_hint": {"type": f"relative_{unit}", "operator": operator, "value": int(value)},
        }

    between_match = BETWEEN_DATES_PATTERN.search(query)
    if between_match:
        start, end = between_match.groups()
        return {
            "sparse_weight": 0.30,
            "dense_weight": 0.70,
            "reason": "explicit_range",
            "requires_prefilter": True,
            "filter_hint": {"type": "date_range", "start": start, "end": end},
        }

    if QUOTED_PHRASE_PATTERN.search(query):
        return {"sparse_weight": 0.85, "dense_weight": 0.15, "reason": "quoted_phrase",
                "requires_prefilter": False, "filter_hint": None}

    if ALPHANUMERIC_ID_PATTERN.search(query):
        return {"sparse_weight": 0.70, "dense_weight": 0.30, "reason": "alphanumeric_id",
                "requires_prefilter": False, "filter_hint": None}

    word_count = len(query.split())
    if word_count <= 3:
        return {"sparse_weight": 0.55, "dense_weight": 0.45, "reason": "short_query",
                "requires_prefilter": False, "filter_hint": None}

    return {"sparse_weight": 0.25, "dense_weight": 0.75, "reason": "conversational",
            "requires_prefilter": False, "filter_hint": None}


In [15]:
test_cases = [
    ("tickets older than 30 days",                        True),
    ("orders placed between March and June",              True),
    ("show me invoices from the past 2 months",            True),
    ("VPN error 691",                                      False),
    ("how do I connect to the office wifi from home",      False),
    ('search for "annual leave policy"',                    False),
]

correct = 0
for query, expected_prefilter in test_cases:
    result = route_query_v2(query)
    status = "PASS" if result["requires_prefilter"] == expected_prefilter else "FAIL"
    correct += (status == "PASS")
    print(f"[{status}] '{query}' -> requires_prefilter={result['requires_prefilter']}, "
          f"filter_hint={result.get('filter_hint')}")

print(f"\nAccuracy: {correct}/{len(test_cases)}")
assert correct == len(test_cases), "Not all test cases passed!"
print("All test cases passed.")


[PASS] 'tickets older than 30 days' -> requires_prefilter=True, filter_hint={'type': 'relative_day', 'operator': 'gt', 'value': 30}
[PASS] 'orders placed between March and June' -> requires_prefilter=True, filter_hint={'type': 'date_range', 'start': 'March', 'end': 'June'}
[PASS] 'show me invoices from the past 2 months' -> requires_prefilter=True, filter_hint={'type': 'relative_month', 'operator': 'gt', 'value': 2}
[PASS] 'VPN error 691' -> requires_prefilter=False, filter_hint=None
[PASS] 'how do I connect to the office wifi from home' -> requires_prefilter=False, filter_hint=None
[PASS] 'search for "annual leave policy"' -> requires_prefilter=False, filter_hint=None

Accuracy: 6/6
All test cases passed.


<h2 id="exercise2">  Exercise 2 — Jurisdiction-Aware Legal Pipeline</h2>

In [16]:
from dataclasses import dataclass, field

@dataclass
class Document:
    doc_id: str
    text: str
    metadata: dict = field(default_factory=dict)
    score: float = 0.0


class LegalMultiStagePipeline:
    def __init__(self, corpus: list):
        self.corpus = corpus

    def pre_filter(self, query: str, tenant_id: str, jurisdiction: str = None) -> list:
        filtered = [d for d in self.corpus if d.metadata.get("tenant_id") == tenant_id]
        before = len(filtered)
        if jurisdiction is not None:
            filtered = [d for d in filtered if d.metadata.get("jurisdiction") == jurisdiction]
        print(f"[pre_filter] {len(self.corpus)} total -> {before} after tenant filter "
              f"-> {len(filtered)} after jurisdiction filter ('{jurisdiction}')")
        return filtered

    def retrieve(self, query: str, candidates: list, top_k: int = 5) -> list:
        query_terms = set(query.lower().split())
        scored = []
        for doc in candidates:
            overlap = len(query_terms & set(doc.text.lower().split()))
            scored.append(Document(doc.doc_id, doc.text, doc.metadata, score=overlap))
        ranked = sorted(scored, key=lambda d: d.score, reverse=True)[:top_k]
        print(f"[retrieve] top {len(ranked)} candidates by keyword overlap")
        return ranked

    def rerank(self, query: str, candidates: list, top_n: int = 2) -> list:
        rescored = []
        for doc in candidates:
            length_penalty = min(len(doc.text.split()) / 10.0, 1.0)
            rescored.append(Document(doc.doc_id, doc.text, doc.metadata, score=doc.score * length_penalty))
        ranked = sorted(rescored, key=lambda d: d.score, reverse=True)[:top_n]
        print(f"[rerank] refined to top {len(ranked)} candidates")
        return ranked

    def generate(self, query: str, context: list, jurisdiction: str = None) -> str:
        if jurisdiction is not None:
            for doc in context:
                if doc.metadata.get("jurisdiction") != jurisdiction:
                    raise ValueError(
                        f"Refusing to generate: document '{doc.doc_id}' has jurisdiction "
                        f"'{doc.metadata.get('jurisdiction')}' but query was scoped to '{jurisdiction}'"
                    )
        joined = " | ".join(d.text for d in context)
        return f"Answer to '{query}' grounded in: {joined}"

    def run(self, query: str, tenant_id: str, jurisdiction: str = None) -> str:
        filtered = self.pre_filter(query, tenant_id, jurisdiction)
        candidates = self.retrieve(query, filtered)
        top_docs = self.rerank(query, candidates)
        return self.generate(query, top_docs, jurisdiction)


In [17]:
corpus = [
    Document("d1", "California requires 24 hour notice for tenant entry", {"tenant_id": "firm1", "jurisdiction": "CA"}),
    Document("d2", "California security deposit limit is two months rent", {"tenant_id": "firm1", "jurisdiction": "CA"}),
    Document("d3", "New York requires written notice for lease termination", {"tenant_id": "firm1", "jurisdiction": "NY"}),
    Document("d4", "New York security deposit rules changed in 2019 reform", {"tenant_id": "firm1", "jurisdiction": "NY"}),
    Document("d5", "California eviction moratorium rules during emergencies", {"tenant_id": "firm1", "jurisdiction": "CA"}),
]

pipeline = LegalMultiStagePipeline(corpus)
answer = pipeline.run("what is the security deposit limit", tenant_id="firm1", jurisdiction="CA")
print("\nFinal answer:", answer)
assert "New York" not in answer, "Jurisdiction leak detected!"
print("Jurisdiction isolation verified: no NY content in a CA-scoped answer.")


[pre_filter] 5 total -> 5 after tenant filter -> 3 after jurisdiction filter ('CA')
[retrieve] top 3 candidates by keyword overlap
[rerank] refined to top 2 candidates

Final answer: Answer to 'what is the security deposit limit' grounded in: California security deposit limit is two months rent | California requires 24 hour notice for tenant entry
Jurisdiction isolation verified: no NY content in a CA-scoped answer.


In [18]:
# Bonus verification: confirm the defensive check in generate() actually fires
# if a bug upstream lets a cross-jurisdiction document slip through.
leaky_context = [
    Document("d3", "New York requires written notice for lease termination", {"jurisdiction": "NY"})
]
try:
    pipeline.generate("what is the notice period", leaky_context, jurisdiction="CA")
    print("ERROR: expected a ValueError but none was raised!")
except ValueError as e:
    print(f"Defensive check fired correctly: {e}")


Defensive check fired correctly: Refusing to generate: document 'd3' has jurisdiction 'NY' but query was scoped to 'CA'


<h2 id="exercise3">  Exercise 3 — Tuning the Pipeline Back Within Budget</h2>

In [19]:
import time
import functools

def time_it(stage_name: str, results: dict):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            start = time.perf_counter()
            result = fn(*args, **kwargs)
            elapsed_ms = (time.perf_counter() - start) * 1000
            results[stage_name] = elapsed_ms
            return result
        return wrapper
    return decorator


timings = {}

class TunedRAGPipeline:

    @time_it("retrieval", timings)
    def retrieve(self, query: str):
        time.sleep(0.04)
        return [f"doc_{i}" for i in range(8)]   # tightened from 20 -> 8

    @time_it("rerank", timings)
    def rerank(self, query: str, docs: list):
        per_doc_cost = 0.012
        fixed_overhead = 0.015
        time.sleep(fixed_overhead + per_doc_cost * len(docs))
        return docs[:5]

    @time_it("generation", timings)
    def generate(self, query: str, docs: list):
        time.sleep(0.79)
        return f"Answer to '{query}'"

    def run(self, query: str):
        docs = self.retrieve(query)
        top_docs = self.rerank(query, docs)
        return self.generate(query, top_docs)


BUDGET_MS = {"retrieval": 50, "rerank": 150, "generation": 800}

def assert_within_budget(timings: dict, budget: dict):
    violations = [
        f"{stage}: {actual:.1f}ms > {budget[stage]}ms budget"
        for stage, actual in timings.items()
        if actual > budget[stage]
    ]
    assert not violations, "Latency budget violated:\n" + "\n".join(violations)


In [20]:
pipeline = TunedRAGPipeline()
answer = pipeline.run("what is our refund policy")

print(f"{'Stage':<12}{'Actual (ms)':<14}{'Budget (ms)':<14}{'Over budget?'}")
for stage, actual_ms in timings.items():
    budget_ms = BUDGET_MS[stage]
    over = "YES" if actual_ms > budget_ms else "no"
    print(f"{stage:<12}{actual_ms:<14.1f}{budget_ms:<14}{over}")

total_actual = sum(timings.values())
total_budget = sum(BUDGET_MS.values())
print(f"\nTotal: {total_actual:.1f}ms actual vs {total_budget}ms budget")

assert_within_budget(timings, BUDGET_MS)
print("All stages within budget.")


Stage       Actual (ms)   Budget (ms)   Over budget?
retrieval   40.1          50            no
rerank      111.1         150           no
generation  790.1         800           no

Total: 941.4ms actual vs 1000ms budget
All stages within budget.


# Extensions: Local Models vs. Cloud/API Models

> **The code cells below require either a paid API key (Cloud/API track) or a local model download**

## Extension A — Local Models Approach

**Architecting Robust RAG Pipelines**

The course's labs use pure Python/regex heuristics precisely so they work offline with
zero setup. This section shows how you'd swap in real *local* (self-hosted, no paid API)
tooling once you're ready to go further.

### What Changes, Component by Component

| Course concept | Course's stand-in | Local-model equivalent |
|---|---|---|
| Query routing | Regex heuristics (`route_query`) | Regex heuristics **or** a small local text-classification model |
| Multi-stage pipeline | In-memory `Document` list + keyword overlap | `langchain_community.vectorstores.FAISS` + `langchain_community.retrievers.BM25Retriever` |
| Generation stage | `f"Answer to '{query}' grounded in: ..."` string template | A locally-hosted LLM (Ollama, llama.cpp, or a local HuggingFace pipeline) |

### 1. Query Routing — Keep the Heuristics, Optionally Add a Local Classifier

The course's core lesson — that routing is a classification problem solvable with
deterministic heuristics before you need a trained model — still holds locally. If you
want to go further than regex, a **local** intent classifier avoids any API call:

In [21]:
# OPTIONAL / ILLUSTRATIVE -- requires `transformers` and a model download; not executed here.
from transformers import pipeline

# Runs entirely on your machine after a one-time model download; no API key.
intent_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
)

def classify_intent_local(query: str) -> str:
    labels = ["exact_id_lookup", "date_range_query", "conversational"]
    result = intent_classifier(query, candidate_labels=labels)
    return result["labels"][0]


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [22]:
classify_intent_local("what's the weather?")

'conversational'

In [23]:
classify_intent_local("can you give me docs prior to 12/31 of this year?")

'date_range_query'

In [24]:
classify_intent_local("docs about dogs with doc id 123")

'exact_id_lookup'

**Trade-off:** this adds a real model load (a few hundred MB to a few GB) and CPU
inference latency (tens to hundreds of milliseconds per query) in exchange for
classification that generalizes beyond what your regex patterns anticipated. For most
teams, the course's heuristic router is still the right starting point — reach for this
only once you have evidence the heuristics are misclassifying a meaningful fraction of
real traffic.

### 2. Multi-Stage Pipeline — Real Local Retrieval

In [26]:
# OPTIONAL / ILLUSTRATIVE -- requires `langchain-huggingface` and `langchain-community`; not executed here.
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Pre-filter first, exactly as the course's pipeline sequencing teaches --
# only embed/index the documents that survive the metadata filter.
TENANT_ID = 'firm1'
tenant_docs = [d for d in corpus if d.metadata.get("tenant_id") == TENANT_ID]

dense_store = FAISS.from_texts(
    [d.text for d in tenant_docs],
    embeddings,
    metadatas=[d.metadata for d in tenant_docs],
)
dense_retriever = dense_store.as_retriever(search_kwargs={"k": 2})

sparse_retriever = BM25Retriever.from_texts([d.text for d in tenant_docs])
sparse_retriever.k = 2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [33]:
corpus

[Document(doc_id='d1', text='California requires 24 hour notice for tenant entry', metadata={'tenant_id': 'firm1', 'jurisdiction': 'CA'}, score=0.0),
 Document(doc_id='d2', text='California security deposit limit is two months rent', metadata={'tenant_id': 'firm1', 'jurisdiction': 'CA'}, score=0.0),
 Document(doc_id='d3', text='New York requires written notice for lease termination', metadata={'tenant_id': 'firm1', 'jurisdiction': 'NY'}, score=0.0),
 Document(doc_id='d4', text='New York security deposit rules changed in 2019 reform', metadata={'tenant_id': 'firm1', 'jurisdiction': 'NY'}, score=0.0),
 Document(doc_id='d5', text='California eviction moratorium rules during emergencies', metadata={'tenant_id': 'firm1', 'jurisdiction': 'CA'}, score=0.0)]

In [35]:
sparse_retriever.invoke('what is the security deposit limit for NYC')

[Document(metadata={}, page_content='California security deposit limit is two months rent'),
 Document(metadata={}, page_content='New York security deposit rules changed in 2019 reform')]

In [32]:
dense_retriever.invoke('what is the security deposit limit for NYC')

[Document(id='ac26e094-024b-494f-9736-5af3fd119b84', metadata={'tenant_id': 'firm1', 'jurisdiction': 'NY'}, page_content='New York security deposit rules changed in 2019 reform'),
 Document(id='eea20b86-5420-4ce9-beb2-5fb06b6a104b', metadata={'tenant_id': 'firm1', 'jurisdiction': 'CA'}, page_content='California security deposit limit is two months rent')]

In [41]:
sparse_retriever.invoke(' 24 hour notice')

[Document(metadata={}, page_content='California requires 24 hour notice for tenant entry'),
 Document(metadata={}, page_content='New York requires written notice for lease termination')]

In [40]:
dense_retriever.invoke('24 hour notice')

[Document(id='c5145460-6f7c-44b0-bbef-de7a7cd2b588', metadata={'tenant_id': 'firm1', 'jurisdiction': 'CA'}, page_content='California requires 24 hour notice for tenant entry'),
 Document(id='dca81a5d-053a-4e17-a86c-04436022f059', metadata={'tenant_id': 'firm1', 'jurisdiction': 'NY'}, page_content='New York requires written notice for lease termination')]

Reranking and fusion for this configuration are covered in the Lesson 3 extension —
Lesson 1's job is the pipeline *shape*, not the retrieval algorithm itself.

### 3. Generation — A Local LLM Instead of a String Template

In [56]:
import os
import subprocess
import time
import requests

OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_LOG = "/tmp/ollama.log"


def ollama_is_running() -> bool:
    try:
        response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=2)
        return response.ok
    except requests.RequestException:
        return False


if not ollama_is_running():
    ollama_log_file = open(OLLAMA_LOG, "w")

    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=ollama_log_file,
        stderr=subprocess.STDOUT,
        env={
            **os.environ,
            "OLLAMA_HOST": "127.0.0.1:11434",
        },
    )

    # Wait for the API server to become available.
    for _ in range(60):
        if ollama_is_running():
            break
        time.sleep(1)
    else:
        with open(OLLAMA_LOG) as log:
            print(log.read())
        raise RuntimeError("Ollama failed to start.")

print("Ollama is running.")

Ollama is running.


In [58]:
!ollama pull llama3.2:1b

In [59]:
# OPTIONAL / ILLUSTRATIVE -- requires `langchain-community` and a running local Ollama server; not executed here.
from langchain_community.llms import Ollama

local_llm = Ollama(model="llama3.2:1b")  # requires `ollama pull llama3.1:8b` once, no API key

def generate_local(query: str, context_docs: list) -> str:
    context = "\n".join(d.page_content for d in context_docs)
    prompt = f"Answer the question using only the context below.\n\nContext:\n{context}\n\nQuestion: {query}"
    return local_llm.invoke(prompt)


In [60]:
dense_retriever.invoke('what is the notice period')

[Document(id='c5145460-6f7c-44b0-bbef-de7a7cd2b588', metadata={'tenant_id': 'firm1', 'jurisdiction': 'CA'}, page_content='California requires 24 hour notice for tenant entry'),
 Document(id='dca81a5d-053a-4e17-a86c-04436022f059', metadata={'tenant_id': 'firm1', 'jurisdiction': 'NY'}, page_content='New York requires written notice for lease termination')]

In [61]:
generate_local("what is the notice period", dense_retriever.invoke('what is the notice period'))

"In California, where the request for 24-hour notice for tenant entry applies. In New York, where the requirement for written notice for lease termination applies. Without specific information about the type of tenancy or the circumstances surrounding the landlord-tenant relationship in your situation, I can only provide general guidance.\n\nIf you're a tenant in California and need to enter a rental property, you should provide 24 hours' written notice before entering the premises.\n\nIf you're a landlord in New York and have a tenant breach their lease agreement or want to terminate the tenancy, they must give the tenant written notice of termination, which can be provided via certified mail with return receipt requested."

### What This Buys You, and What It Costs

**Pros of the local approach for Lesson 1's lessons:**
- No API key, no per-call cost, no rate limits — you can run the multi-tenant isolation test from Exercise 2 as many times as you want without a bill
- Fully deterministic if you fix the model's random seed, which keeps the spirit of the course's `assert`-based verification intact
- No network dependency once models are downloaded — the pipeline keeps working on a plane

**Cons to budget for:**
- A real download requirement (embedding model + LLM weights) that the course's zero-setup labs deliberately avoid
- Meaningfully slower generation on CPU-only hardware — this directly changes Lesson 1's latency budget math. A local 8B-parameter model on a laptop CPU can easily take several seconds per response, blowing through the ~800ms generation budget the course's `NaiveRAGPipeline` example targets. On local hardware, treat the entire latency budget table as needing re-derivation against your own measured hardware, not the course's illustrative numbers.
- Quality is usually lower than a frontier hosted model, which matters more for generation than for retrieval

### Bridging Back to the Course

The pipeline *sequencing* lesson (pre-filter → retrieve → rerank → generate) and the
*latency budgeting* discipline (measure, don't assume) both transfer directly. What
changes is only the concrete numbers — profile your own stages with the course's
`time_it` decorator against these real local components rather than assuming the
course's simulated timings apply.

---

## Extension B — Cloud/API Models Approach

**Architecting Robust RAG Pipelines**

The course's labs use pure Python/regex heuristics precisely so they work offline with
zero setup, zero cost, and zero rate-limit risk for a live cohort. This section shows
how you'd swap in real *cloud/API* tooling once you're ready to go further.

### What Changes, Component by Component

| Course concept | Course's stand-in | Cloud/API equivalent |
|---|---|---|
| Query routing | Regex heuristics (`route_query`) | Structured-output classification via `ChatOpenAI` + `pydantic` |
| Multi-stage pipeline | In-memory `Document` list + keyword overlap | `FAISS` (still local storage) with `OpenAIEmbeddings` for the vectors themselves |
| Generation stage | `f"Answer to '{query}' grounded in: ..."` string template | `langchain_openai.ChatOpenAI` (e.g. `gpt-4o-mini`) |

### 1. Query Routing — Structured Output Instead of Regex

In [63]:
# OPTIONAL / ILLUSTRATIVE -- requires `langchain-openai`, `pydantic`, and a funded OPENAI_API_KEY; not executed here.
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

class RouteDecision(BaseModel):
    reason: str = Field(description="One of: alphanumeric_id, quoted_phrase, relative_range, conversational")
    requires_prefilter: bool = Field(description="True if the query implies a date/numeric range filter")
    sparse_weight: float = Field(description="0-1 weight for sparse retrieval")
    dense_weight: float = Field(description="0-1 weight for dense retrieval")

router_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(RouteDecision)

router_prompt = ChatPromptTemplate.from_template(
    "Classify this search query for a retrieval router: {query}"
)

def route_query_cloud(query: str) -> dict:
    decision = router_llm.invoke(router_prompt.format(query=query))
    return decision.model_dump()


In [64]:
route_query_cloud("what is the weather?")

{'reason': 'conversational',
 'requires_prefilter': False,
 'sparse_weight': 0.2,
 'dense_weight': 0.8}

In [65]:
route_query_cloud("can you give me docs prior to 12/31 of this year?")

{'reason': 'relative_range',
 'requires_prefilter': True,
 'sparse_weight': 0.2,
 'dense_weight': 0.8}

In [66]:
route_query_cloud("docs that have `24 hour notice`")

{'reason': 'quoted_phrase',
 'requires_prefilter': False,
 'sparse_weight': 0.3,
 'dense_weight': 0.7}

**Trade-off:** this is strictly more flexible than regex — it will correctly classify
phrasings the course's patterns never anticipated — but every single query now costs a
network round trip and a token-billed API call, and `temperature=0` reduces but does not
eliminate output variance. The course deliberately avoids this for its labs because a
cohort of ~25 people all calling this per-query router simultaneously is a realistic way
to hit rate limits mid-exercise — exactly the concern flagged in Lesson 4's Ragas
exercise about `429` errors under batch load.

### 2. Multi-Stage Pipeline — Cloud Embeddings, Local Index

In [68]:
# OPTIONAL / ILLUSTRATIVE -- requires `langchain-openai`, `langchain-community`, and a funded OPENAI_API_KEY; not executed here.
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

tenant_docs = [d for d in corpus if d.metadata.get("tenant_id") == TENANT_ID]
dense_store = FAISS.from_texts(
    [d.text for d in tenant_docs],
    embeddings,
    metadatas=[d.metadata for d in tenant_docs],
)


Note that FAISS itself is still a local, in-process index either way — "cloud" here
refers to where the *embedding computation* happens, not where vectors are stored. If
you want a fully managed cloud vector store as well (removing local index management
entirely), that's a Lesson 2 concern (Pinecone, Weaviate Cloud, etc.), not a Lesson 1 one.

### 3. Generation — A Real Hosted LLM

In [69]:
# OPTIONAL / ILLUSTRATIVE -- requires `langchain-openai` and a funded OPENAI_API_KEY; not executed here.
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

generator = ChatOpenAI(model="gpt-4o-mini", temperature=0)
generation_prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context below.\n\nContext:\n{context}\n\nQuestion: {query}"
)
generation_chain = generation_prompt | generator | StrOutputParser()

def generate_cloud(query: str, context_docs: list) -> str:
    context = "\n".join(d.page_content for d in context_docs)
    return generation_chain.invoke({"context": context, "query": query})


In [71]:
generate_cloud("what is the notice period", dense_retriever.invoke('what is the notice period'))

'The notice period for tenant entry in California is 24 hours. In New York, written notice is required for lease termination, but the specific notice period is not provided in the context.'

### What This Buys You, and What It Costs

**Pros of the cloud approach for Lesson 1's lessons:**
- No local hardware or model-download requirement — works identically on any laptop
- Highest-quality routing and generation available, closest to what you'll actually ship
- Zero maintenance of local model weights or GPU drivers

**Cons to budget for — all directly relevant to what Lesson 1 teaches:**
- **Latency budgeting gets harder, not easier.** The course's illustrative ~800ms generation budget assumed a fast hosted model; real network latency, provider-side queueing, and token-by-token generation time all now count against that budget, and vary run to run in a way the course's deterministic `time.sleep()` stand-ins never do. Re-profile with the course's `time_it` decorator against your real endpoint before trusting any budget number.
- **Rate limits are a first-class system constraint now**, not a footnote. A router or generator call per query, multiplied by concurrent users, is exactly the kind of load Lesson 4 warns about hitting `429` errors under.
- **Cost accrues per call.** Routing every query through an LLM (rather than free regex) adds a real, ongoing line item that scales with traffic — worth weighing against the marginal accuracy gain over the heuristic router for your actual query distribution.
- **Non-determinism** means the multi-tenant isolation assertion from Exercise 2 (`assert "New York" not in answer`) is still valid as a hard safety check, but softer correctness checks elsewhere may need to tolerate wording variance across runs.

### Bridging Back to the Course

The pipeline *sequencing* lesson and the discipline of *measuring* a latency budget
rather than assuming one both transfer directly — arguably they matter **more** once
real network calls are in the loop, not less. What changes is that Lesson 1's
"generation dominates the budget" finding becomes even more pronounced with a real
hosted model, and the router itself now has a real cost/latency profile worth including
in that budget, which the course's free heuristic router never needed to account for.